# 02 — Advanced feature engineering

## What this notebook teaches

- A feature must be available when the prediction is made. `duration` is excluded because it is
  known only during or after the call.
- Every learned transformation must be fit inside a cross-validation pipeline.
- A plausible feature is kept only when ablation shows a useful, stable improvement.

We compare a one-hot logistic-regression baseline with CatBoost. This is a workflow comparison:
a better CatBoost score does not isolate native categorical handling as the cause.


In [15]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# Known interoperability/UI warnings do not affect predictions or notebook execution.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # Return the course root when present, otherwise the notebook's directory.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # Set FAST_MODE=0 for full-size experiments; laptop mode is the default.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # The course ships with a local dataset; notebooks never access the network.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # Load the data, encode y, and exclude post-call duration by default.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # Deterministic stratified 60/20/20 split; test stays sealed until notebook 09.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # Preprocessing is fitted only inside the enclosing training/CV pipeline.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute probability-quality, ranking, and threshold metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y_true, prediction),
            "log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "average_precision": average_precision_score(y_true, probability),
            "roc_auc": roc_auc_score(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan}

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
BRAND_COLOR = "#2a9d8f"
ACCENT_COLOR = "#e76f51"
sns.set_theme(style="whitegrid", context="notebook", palette=[BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"])
plt.rcParams.update({
    "axes.facecolor": "#fcfcfc",
    "figure.facecolor": "white",
    "axes.edgecolor": "#d9d9d9",
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})


{'FAST_MODE': True, 'CV_FOLDS': 3, 'seed': 42}


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay)

development, validation, _sealed_test = make_splits(load_bank_data(), reduced=FAST_MODE)
X_dev, y_dev = split_xy(development)
X_val, y_val = split_xy(validation)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
CV_SPLITS = list(cv.split(X_dev, y_dev))


## Ordered target statistics

The next cell demonstrates CatBoost's key categorical safety mechanism. For each row, the
category statistic uses only earlier rows in a permutation; a prior stabilizes sparse categories.
Naive full-data target encoding would leak the current row's label.


In [17]:
toy_order = pd.DataFrame({
    "step": np.arange(1, 9),
    "job": ["admin", "services", "admin", "admin",
            "services", "student", "admin", "student"],
    "subscribed": [1, 0, 0, 1, 1, 1, 0, 0],
})
prior, prior_weight = 0.5, 2.0
category_sums, category_counts = {}, {}
ordered_values, histories = [], []
for row in toy_order.itertuples(index=False):
    previous_sum = category_sums.get(row.job, 0)
    previous_count = category_counts.get(row.job, 0)
    ordered_values.append(
        (previous_sum + prior_weight * prior) / (previous_count + prior_weight)
    )
    histories.append(previous_count)
    category_sums[row.job] = previous_sum + row.subscribed
    category_counts[row.job] = previous_count + 1

toy_order["ordered_stat"] = ordered_values
toy_order["earlier_same_job_rows"] = histories
toy_order["naive_full_data_stat"] = toy_order.groupby("job")["subscribed"].transform("mean")
job_colors = dict(zip(toy_order["job"].unique(), sns.color_palette("Set2", 3)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(toy_order["step"], toy_order["subscribed"], s=110,
                c=[job_colors[j] for j in toy_order["job"]], edgecolor="#4a4a4a")
for row in toy_order.itertuples(index=False):
    axes[0].annotate(row.job, (row.step, row.subscribed), xytext=(0, 9),
                     textcoords="offset points", ha="center", fontsize=8)
axes[0].set(xticks=toy_order["step"], yticks=[0, 1], yticklabels=["no", "yes"],
            xlabel="position in permutation", ylabel="observed label",
            title="1. Rows arrive in order")

for job, group in toy_order.groupby("job", sort=False):
    axes[1].plot(group["step"], group["ordered_stat"], marker="o", linewidth=2,
                 color=job_colors[job], label=f"{job}: ordered")
    axes[1].plot(group["step"], group["naive_full_data_stat"], linestyle="--",
                 linewidth=1.4, color=job_colors[job], alpha=0.65)
axes[1].axhline(prior, color="#4a4a4a", linestyle=":", label="prior = 0.5")
axes[1].set(xlabel="position in permutation", ylabel="encoded value", ylim=(-0.05, 1.05),
            title="2. Encode before revealing this label")
axes[1].legend(fontsize=7, ncol=2)

bars = axes[2].bar(toy_order["step"], toy_order["earlier_same_job_rows"],
                   color=[job_colors[j] for j in toy_order["job"]], edgecolor="#4a4a4a")
axes[2].bar_label(bars, fontsize=8)
axes[2].set(xticks=toy_order["step"], xlabel="position in permutation",
            ylabel="earlier rows with same job",
            title="3. Evidence grows without peeking ahead")
fig.suptitle("Ordered category statistics use only the past of the permutation", y=1.04)
plt.tight_layout()
plt.show()
display(toy_order.round(3))


,step,job,subscribed,ordered_stat,earlier_same_job_rows,naive_full_data_stat
0,1,admin,1,0.500,0,0.5
1,2,services,0,0.500,0,0.5
2,3,admin,0,0.667,1,0.5
3,4,admin,1,0.500,2,0.5
4,5,services,1,0.333,1,0.5
5,6,student,1,0.500,0,0.5
6,7,admin,0,0.600,3,0.5
7,8,student,0,0.667,1,0.5


## Audit the raw feature space

Inspect data types, cardinality, missingness, and sentinel values before designing features.
Here, `pdays=-1` means "not previously contacted"; it is a state to model, not a numeric
recency value or an ordinary missing value.


In [18]:
feature_audit = pd.DataFrame({
    "dtype": X_dev.dtypes.astype(str),
    "unique": X_dev.nunique(dropna=False),
    "missing_pct": X_dev.isna().mean().mul(100),
}).sort_values(["dtype", "unique"])
display(feature_audit)
display(pd.crosstab(X_dev["pdays"].eq(-1), y_dev, normalize="index")
        .rename(index={True: "never contacted", False: "contacted"}))


,dtype,unique,missing_pct
day,int64,31,0.0
previous,int64,33,0.0
campaign,int64,40,0.0
age,int64,74,0.0
pdays,int64,409,0.0
balance,int64,4077,0.0
default,object,2,0.0
housing,object,2,0.0
loan,object,2,0.0
marital,object,3,0.0


y,0,1
pdays,,
contacted,0.782407,0.217593
never contacted,0.905081,0.094919


## Add testable feature hypotheses

- Split `pdays=-1` into a previous-contact indicator and nullable recency.
- Test contact pressure, `campaign / (1 + previous)`.
- Test a weak ratio, `balance / age`, and age bands for nonlinearity.

These are predictive hypotheses, not causal claims. The transformer is row-wise and stateless;
imputation, scaling, and category vocabularies are learned later inside the pipeline.


In [19]:
REQUIRED_FEATURE_COLUMNS = {"pdays", "campaign", "previous", "balance", "age"}

def add_domain_features(frame):
    """Create pre-contact domain features using row-wise, stateless logic."""
    missing = REQUIRED_FEATURE_COLUMNS.difference(frame.columns)
    if missing:
        raise ValueError(f"missing required columns: {sorted(missing)}")
    if frame["age"].dropna().le(0).any():
        raise ValueError("age must be positive when present")
    if frame["previous"].dropna().lt(0).any():
        raise ValueError("previous must be non-negative when present")
    result = frame.copy()

    # In this dataset, pdays=-1 means the client was never previously contacted.
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)

    # Hypotheses to validate by ablation rather than assume useful.
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(
        result["age"],
        bins=[0, 29, 39, 49, 59, np.inf],
        labels=["<30", "30s", "40s", "50s", "60+"],
    ).astype("object")

    return result.drop(columns=["pdays"])


In [20]:
engineered_preview = add_domain_features(X_dev.head())
engineered_preview.T


,0,1,2,3,4
age,40,56,56,31,30
job,blue-collar,management,admin.,self-employed,management
marital,married,married,divorced,single,single
education,secondary,tertiary,secondary,tertiary,tertiary
default,no,no,no,no,no
balance,271,1282,-429,283,61
housing,yes,no,no,no,yes
loan,no,no,yes,no,no
contact,unknown,cellular,cellular,cellular,telephone
day,29,19,23,28,5


## Inspect the engineered features

This diagnostic view checks that each transformation has the intended meaning. It does not show
that a feature improves the model; cross-validation and ablation answer that question.


In [21]:
process_sample = X_dev.sample(min(2500, len(X_dev)), random_state=SEED)
process_features = add_domain_features(process_sample)
fig, axes = plt.subplots(2, 2, figsize=(13, 8), dpi=140)

contacted_counts = process_features["was_previously_contacted"].value_counts().reindex([0, 1])
axes[0, 0].bar(["never contacted\n(raw pdays = -1)", "previously contacted"],
               contacted_counts, color=["#9aa0a6", BRAND_COLOR])
axes[0, 0].set(ylabel="rows", title="1. Separate state from recency")
axes[0, 0].text(0.5, 0.92, "pdays_clean retains recency only when it exists",
                transform=axes[0, 0].transAxes, ha="center", fontsize=9)

pressure_view = process_features.loc[:, ["campaign", "previous", "contact_pressure"]].copy()
pressure_view["previous history"] = np.where(
    pressure_view["previous"].eq(0), "no previous contacts", "has previous contacts"
)
sns.scatterplot(data=pressure_view, x="campaign", y="contact_pressure",
                hue="previous history", palette=[BRAND_COLOR, ACCENT_COLOR],
                alpha=0.45, s=28, ax=axes[0, 1])
axes[0, 1].set(title="2. Normalize campaign effort by history",
               ylabel="campaign / (1 + previous)")

lower, upper = process_features["balance_per_age"].quantile([0.01, 0.99])
ratio_view = process_features["balance_per_age"].clip(lower, upper)
axes[1, 0].scatter(process_sample["age"], ratio_view, s=12, alpha=0.25,
                   color=BRAND_COLOR)
axes[1, 0].axhline(0, color="#4a4a4a", linewidth=1)
axes[1, 0].set(xlabel="age", ylabel="balance / age (1st-99th pct shown)",
               title="3. Express balance relative to age")

age_counts = process_features["age_band"].value_counts().reindex(
    ["<30", "30s", "40s", "50s", "60+"]
)
axes[1, 1].bar(
    age_counts.index, age_counts.values,
    color=sns.color_palette([BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261", "#9aa0a6"]),
)
axes[1, 1].set(xlabel="engineered age band", ylabel="rows",
               title="4. Turn continuous age into regions")

fig.suptitle("Raw columns become explicit, testable feature hypotheses", y=1.01, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Encode categories and choose metrics

`OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10)` groups rare training-fold
categories and handles unseen values. It returns a sparse matrix; do not densify high-cardinality
features without checking memory.

The positive class is uncommon, so accuracy alone is misleading. Report:

- **balanced accuracy** for equal attention to both classes;
- **precision** for the quality of positive predictions;
- **recall** for the share of positives found;
- **F1** for a compact precision–recall balance.

These metrics depend on a threshold. We use `0.5` for comparison; production should choose it
from business cost and capacity, not after inspecting validation results.


In [22]:
positive_rate = y_dev.mean()
majority_accuracy = max(positive_rate, 1 - positive_rate)
print(f"positive class: {positive_rate:.1%}")
print(f"always predict 'no' accuracy: {majority_accuracy:.1%}")


positive class: 11.7%
always predict 'no' accuracy: 88.3%


## Build a fold-safe preprocessing pipeline

Numeric columns are median-imputed and standardized. Categorical columns are imputed and
one-hot encoded. Keeping all learned steps inside `Pipeline` ensures that each CV fold learns
its statistics from training rows only.


In [23]:
def make_preprocessor(frame):
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()

    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=10,
            sparse_output=True,
        )),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipe, numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)


In [24]:
baseline = Pipeline([
    ("preprocess", make_preprocessor(development)),
    ("model", LogisticRegression(max_iter=1200, random_state=SEED)),
])
engineered_frame = add_domain_features(development)
engineered = Pipeline([
    ("features", FunctionTransformer(add_domain_features, validate=False)),
    ("preprocess", make_preprocessor(engineered_frame)),
    ("model", LogisticRegression(max_iter=1200, random_state=SEED)),
])

fold_metrics = {}
CV_SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "log_loss": "neg_log_loss",
    "brier_score": "neg_brier_score",
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
}
NEGATED_METRICS = {"log_loss", "brier_score"}

def cv_summary(name, estimator):
    scores = cross_validate(
        estimator, X_dev, y_dev, cv=CV_SPLITS,
        scoring=CV_SCORING,
        n_jobs=-1,
    )
    fold_metrics[name] = pd.DataFrame({
        metric: scores[f"test_{metric}"] * (-1 if metric in NEGATED_METRICS else 1)
        for metric in CV_SCORING
    }).rename_axis("fold")
    summary = {
        "experiment": name,
        "accuracy_mean": fold_metrics[name]["accuracy"].mean(),
        "accuracy_sd": fold_metrics[name]["accuracy"].std(ddof=1),
        "accuracy_se": fold_metrics[name]["accuracy"].sem(ddof=1),
    }
    summary.update({
        metric: fold_metrics[name][metric].mean()
        for metric in CV_SCORING if metric != "accuracy"
    })
    return summary

ablation = pd.DataFrame([cv_summary("raw features", baseline),
                         cv_summary("domain features", engineered)]).set_index("experiment")
ablation


,accuracy_mean,accuracy_sd,accuracy_se,balanced_accuracy,precision,recall,f1
experiment,,,,,,,
raw features,0.888083,0.000878,0.000507,0.562812,0.594400,0.138177,0.224104
domain features,0.887500,0.000661,0.000382,0.564954,0.578066,0.143875,0.230178


## Ablate feature families, not individual columns

Compare raw features, all engineered features, and one removed family at a time. This tests the
value of contact history, contact pressure, and demographics while keeping preprocessing inside
each CV fold. Small differences across a few folds are inconclusive; operational cost matters too.


In [25]:
FEATURE_FAMILIES = {
    "contact history": ["was_previously_contacted", "pdays_clean"],
    "contact pressure": ["contact_pressure"],
    "demographic": ["balance_per_age", "age_band"],
}

def domain_features_without(frame, excluded=()):
    return add_domain_features(frame).drop(columns=list(excluded), errors="ignore")

def pipeline_without(excluded=()):
    schema = domain_features_without(development, excluded)
    transform = FunctionTransformer(
        domain_features_without, kw_args={"excluded": tuple(excluded)}, validate=False
    )
    return Pipeline([
        ("features", transform),
        ("preprocess", make_preprocessor(schema)),
        ("model", LogisticRegression(max_iter=1200, random_state=SEED)),
    ])

family_ablation = [cv_summary("all engineered", pipeline_without())]
for family, columns in FEATURE_FAMILIES.items():
    family_ablation.append(cv_summary(f"without {family}", pipeline_without(columns)))
family_ablation = pd.DataFrame(family_ablation).set_index("experiment")
family_ablation["delta_accuracy"] = (
    family_ablation["accuracy_mean"] - family_ablation.loc["all engineered", "accuracy_mean"]
)
family_ablation["delta_balanced_accuracy"] = (
    family_ablation["balanced_accuracy"]
    - family_ablation.loc["all engineered", "balanced_accuracy"]
)
display(family_ablation.sort_values("accuracy_mean", ascending=False))

all_folds = fold_metrics["all engineered"]
paired_rows = []
for experiment in family_ablation.index.drop("all engineered"):
    difference = fold_metrics[experiment] - all_folds
    paired_rows.append({
        "experiment": experiment,
        **{f"delta_{metric}": difference[metric].mean()
           for metric in ["accuracy", "balanced_accuracy", "recall", "f1"]},
    })
paired_differences = pd.DataFrame(paired_rows).set_index("experiment")
display(paired_differences)

plot_data = pd.concat(
    {name: values[["accuracy", "balanced_accuracy"]]
     for name, values in fold_metrics.items()
     if name == "all engineered" or name.startswith("without")},
    names=["experiment", "fold"],
).reset_index().melt(
    id_vars=["experiment", "fold"], var_name="metric", value_name="score"
)
g = sns.catplot(
    data=plot_data, x="score", y="experiment", hue="fold", col="metric",
    kind="point", sharex=False, height=3.5, aspect=1.15,
    palette=[BRAND_COLOR, ACCENT_COLOR, "#264653"],
)
g.set_titles("{col_name}")
g.fig.suptitle(
    "Paired fold results: engineered feature-family ablation",
    y=1.05, fontsize=14, fontweight="bold",
)
plt.show()


,accuracy_mean,accuracy_sd,accuracy_se,balanced_accuracy,precision,recall,f1,delta_accuracy,delta_balanced_accuracy
experiment,,,,,,,,,
without demographic,0.888000,0.000866,0.000500,0.563383,0.591724,0.139601,0.225738,0.000500,-0.001571
without contact pressure,0.887667,0.000382,0.000220,0.565666,0.580050,0.145299,0.232181,0.000167,0.000712
all engineered,0.887500,0.000661,0.000382,0.564954,0.578066,0.143875,0.230178,0.000000,0.000000
without contact history,0.887250,0.000661,0.000382,0.564194,0.573329,0.142450,0.228064,-0.000250,-0.000759


,delta_accuracy,delta_balanced_accuracy,delta_recall,delta_f1
experiment,,,,
without contact history,-0.000250,-0.000759,-0.001425,-0.002113
without contact pressure,0.000167,0.000712,0.001425,0.002004
without demographic,0.000500,-0.001571,-0.004274,-0.004440


/Users/soroush/Library/Python/3.9/lib/python/site-packages/seaborn/_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/Users/soroush/Library/Python/3.9/lib/python/site-packages/seaborn/_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/Users/soroush/Library/Python/3.9/lib/python/site-packages/seaborn/categorical.py:1200: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explic

## Test feature code

Feature code is production code. Test schema requirements, row preservation, null behavior,
finite ratios, valid bands, and the rule that `duration` never enters the model.


In [26]:
check = add_domain_features(X_dev)
assert len(check) == len(X_dev)
assert check.index.equals(X_dev.index)
assert "duration" not in check
assert np.isfinite(check["contact_pressure"]).all()
assert np.isfinite(check["balance_per_age"]).all()
assert check["age_band"].notna().all()
original = X_dev.head(2).copy(deep=True)
add_domain_features(original)
pd.testing.assert_frame_equal(original, X_dev.head(2))

empty = add_domain_features(X_dev.iloc[:0])
assert empty.empty and list(empty.columns) == list(check.columns)

with_null = X_dev.head(1).copy()
with_null.loc[with_null.index[0], "age"] = np.nan
null_result = add_domain_features(with_null)
assert pd.isna(null_result["balance_per_age"].iloc[0])
assert pd.isna(null_result["age_band"].iloc[0])

try:
    add_domain_features(X_dev.drop(columns="age"))
    raise AssertionError("missing-column validation did not run")
except ValueError as error:
    assert "age" in str(error)

invalid_age = X_dev.head(1).copy()
invalid_age["age"] = 0
try:
    add_domain_features(invalid_age)
    raise AssertionError("age validation did not run")
except ValueError as error:
    assert "positive" in str(error)

print("feature checks passed, including schema, null, empty, and mutation cases")


feature checks passed, including schema, null, empty, and mutation cases


## Inspect the transformed feature space

Inspect output names and feature count to catch omissions or category explosions. Logistic
coefficients describe associations in this fitted model, not causal effects; correlated one-hot
features can make individual coefficients unstable.


In [27]:
inspection_model = engineered.fit(X_dev, y_dev)
names = inspection_model.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.Series(
    inspection_model.named_steps["model"].coef_[0], index=names, name="coefficient"
)
display(pd.concat([coefficients.nsmallest(10), coefficients.nlargest(10)]).to_frame())
print(f"{len(X_dev.columns)} raw columns -> {len(names)} transformed columns")

unseen = X_val.head(1).copy()
unseen["job"] = "__NEW_JOB_AT_INFERENCE__"
transformed_unseen = inspection_model[:-1].transform(unseen)
assert transformed_unseen.shape[1] == len(names)
print("unseen category transformed safely; output shape", transformed_unseen.shape)


,coefficient
categorical__contact_unknown,-1.063268
categorical__month_jan,-1.018041
categorical__month_nov,-0.871529
categorical__poutcome_failure,-0.832912
categorical__month_aug,-0.737569
categorical__job_housemaid,-0.710218
categorical__month_jul,-0.613841
categorical__poutcome_unknown,-0.481374
numeric__campaign,-0.477397
categorical__housing_yes,-0.468155


15 raw columns -> 58 transformed columns
unseen category transformed safely; output shape (1, 58)


## Compare with CatBoost

CatBoost receives raw categorical strings and uses ordered target statistics rather than a
full-data target encoder. In every outer CV fold, its stopping iteration is selected only from
that fold's training rows before refitting and scoring the untouched fold.

CatBoost and logistic regression therefore share the same outer folds and report probability
quality (log loss, Brier score), ranking (PR-AUC, ROC-AUC), and threshold metrics.


In [28]:
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

class EarlyStoppedCatBoostClassifier(ClassifierMixin, BaseEstimator):
    """Tune CatBoost stopping within each fitted training fold, then refit it."""

    def __init__(self, iterations, depth=6, learning_rate=0.06,
                 validation_fraction=0.15, early_stopping_rounds=40,
                 random_seed=SEED, thread_count=1):
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.validation_fraction = validation_fraction
        self.early_stopping_rounds = early_stopping_rounds
        self.random_seed = random_seed
        self.thread_count = thread_count

    def _prepare(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("CatBoost requires a pandas DataFrame with named columns")
        missing = set(self.feature_names_in_).difference(X.columns)
        if missing:
            raise ValueError(f"missing feature columns: {sorted(missing)}")
        prepared = X.loc[:, self.feature_names_in_].copy()
        for column in self.cat_features_:
            prepared[column] = prepared[column].fillna("__MISSING__").astype(str)
        return prepared

    def _model(self, iterations, eval_metric=None):
        return CatBoostClassifier(
            iterations=iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            loss_function="Logloss",
            eval_metric=eval_metric,
            random_seed=self.random_seed,
            thread_count=self.thread_count,
            verbose=False,
            allow_writing_files=False,
        )

    def fit(self, X, y):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.cat_features_ = X.select_dtypes(
            include=["object", "category", "bool"]
        ).columns.tolist()
        prepared = self._prepare(X)
        X_fit, X_stop, y_fit, y_stop = train_test_split(
            prepared, y,
            test_size=self.validation_fraction,
            stratify=y,
            random_state=self.random_seed,
        )
        selection_model = self._model(self.iterations, eval_metric="Logloss")
        selection_model.fit(
            X_fit, y_fit,
            cat_features=self.cat_features_,
            eval_set=(X_stop, y_stop),
            early_stopping_rounds=self.early_stopping_rounds,
            verbose=False,
        )
        best_iteration = selection_model.get_best_iteration()
        self.best_iterations_ = (
            self.iterations if best_iteration < 0 else best_iteration + 1
        )
        self.model_ = self._model(self.best_iterations_)
        self.model_.fit(prepared, y, cat_features=self.cat_features_, verbose=False)
        self.classes_ = np.unique(y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._prepare(X))

    def predict(self, X):
        return self.model_.predict(self._prepare(X)).astype(int)

catboost_candidate = EarlyStoppedCatBoostClassifier(
    iterations=180 if FAST_MODE else 450,
)
catboost_cv = pd.DataFrame([
    cv_summary("native CatBoost", catboost_candidate)
]).set_index("experiment")
display(catboost_cv)

cat_model = catboost_candidate.fit(X_dev, y_dev)
print(f"refit CatBoost on all development rows with {cat_model.best_iterations_} trees")

onehot_model = inspection_model
raw_onehot_model = baseline.fit(X_dev, y_dev)
validation_probabilities = {
    "always no": np.zeros(len(y_val)),
    "raw one-hot logistic": raw_onehot_model.predict_proba(X_val)[:, 1],
    "engineered one-hot logistic": onehot_model.predict_proba(X_val)[:, 1],
    "native CatBoost": cat_model.predict_proba(X_val)[:, 1],
}
comparison = pd.DataFrame({
    name: classification_metrics(y_val, probability)
    for name, probability in validation_probabilities.items()
}).T
display(comparison)

fig, axes = plt.subplots(1, 4, figsize=(17, 3.5))
for axis, (name, probability) in zip(axes, validation_probabilities.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_val, probability >= 0.5, display_labels=["no", "yes"],
        colorbar=False, cmap=sns.light_palette(BRAND_COLOR, as_cmap=True), ax=axis,
    )
    axis.set_title(name, pad=12, fontweight="bold")
fig.suptitle("Validation confusion matrices at threshold 0.5", y=1.03, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

best_name = comparison["balanced_accuracy"].idxmax()
print(
    f"At threshold 0.5, {best_name} has the highest balanced accuracy "
    f"({comparison.loc[best_name, 'balanced_accuracy']:.3f}), with "
    f"accuracy={comparison.loc[best_name, 'accuracy']:.3f}, "
    f"precision={comparison.loc[best_name, 'precision']:.3f}, and "
    f"recall={comparison.loc[best_name, 'recall']:.3f}."
)


refit CatBoost on all development rows with 176 trees


,accuracy,balanced_accuracy,precision,recall,f1
always no,0.88300,0.500000,0.000000,0.000000,0.000000
raw one-hot logistic,0.89000,0.573475,0.614754,0.160256,0.254237
engineered one-hot logistic,0.88925,0.570270,0.605042,0.153846,0.245315
native CatBoost,0.89450,0.588071,0.676923,0.188034,0.294314


At threshold 0.5, native CatBoost has the highest balanced accuracy (0.588), with accuracy=0.894, precision=0.677, and recall=0.188.


## Interpret the comparison

CatBoost changes both the encoding strategy and the model family, so this is not a pure encoding
experiment. Compare metrics at the fixed threshold, inspect the confusion matrices, and avoid
further tuning on this validation set.

## Senior takeaway

Safe feature engineering starts with a prediction-time contract. Fit learned transformations
inside cross-validation, test feature code, and use ablation—not intuition—to decide whether a
feature is worth its operational complexity.
